In [ ]:
import os
import sys
import platform
import zipfile
from pathlib import Path
from pathlib import Path
from collections import Counter
import importlib
from statistics import mean
import pandas as pd
from IPython.display import display

In [2]:
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "team_a/configs/evaluation.json").is_file())
sys.path.insert(0, str(repo))
from team_a.src.common_runtime import agent_import_path, load_config, sha256, verify_decks, verify_sdk
from team_a.src.models.jiayi.model_1 import Model as Linear
from team_a.src.models.jiayi.model_2 import Model as MLP
from team_a.src.models.jiayi.self_play import SHAPING_SCALE, play_battle

SDK_DIR = os.environ.get("PTCG_SDK_DIR") 
if not SDK_DIR:
    print("Blocked: set PTCG_SDK_DIR to the official sample_submission directory containing cg/ and deck.csv.")
    print("No native battles executed.")

In [3]:
if SDK_DIR:
    sdk_path = Path(SDK_DIR).expanduser().resolve()
    config = load_config()
    sdk_hashes = verify_sdk(sdk_path, config)
    deck_path = sdk_path / "deck.csv"
    deck = [int(line) for line in deck_path.read_text().splitlines() if line.strip()]
    if len(deck) != 60:
        raise ValueError("Expected the official 60-card sample deck")
    deck_id = {"deck_sha256": sha256(deck_path)}
    verify_decks(deck_id, deck_id, config)
    model_dir = repo / "team_a/src/models/jiayi"
    specs = [("A-J1", Linear, model_dir / "aj1_bc.json"),
             ("A-J2", MLP, model_dir / "aj2_bc.json")]
    epsilon = 0.1
    seeds = (61000, 61001, 61002, 61003)
    print("Python", sys.version.split()[0], "platform", sys.platform)
    print("Seeds", seeds, "epsilon", epsilon, "shaping scale", SHAPING_SCALE)
    print("Deck SHA-256", deck_id["deck_sha256"])
    print("SDK hashes", sdk_hashes)
    print("Checkpoint hashes", {name: sha256(path) for name, _, path in specs})
    battles, decisions = [], []
    with agent_import_path(sdk_path):
        for name in ("cg.game", "cg.sim"):
            module = importlib.import_module(name)
            expected = sdk_path / (name.replace(".", "/") + ".py")
            if Path(module.__file__).resolve() != expected:
                raise RuntimeError("Restart the kernel to load cg from the configured SDK directory")
        for name, loader, path in specs:
            model = loader(path)
            for index, seed in enumerate(seeds):
                result, trace = play_battle((model, model), deck, seed, epsilon, bool(index % 2))
                counts = Counter(d["source"] for d in trace)
                battles.append(dict(result, policy=name, learned=counts["learned"], fallback=counts["heuristic"]))
                decisions.extend(dict(d, policy=name, seed=seed) for d in trace)
    display(pd.DataFrame(battles).reindex(columns=["policy", "python_seed", "reverse_player", "status", "result_code",
                                                    "steps", "learned", "fallback", "terminal_returns", "shaped_returns",
                                                    "invalid_responses", "engine_rejections", "error_message"]))

Python 3.13.18 platform linux
Seeds (61000, 61001, 61002, 61003) epsilon 0.1 shaping scale 0.25
Deck SHA-256 42068a1803902756badcfd418f6f348b7901365a281d78af0692cbf2589f0799
SDK hashes {'cg/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cg/api.py': '593f1298e52a635f90f8f505a52113e9af114f444c293404e37906f18ee06ced', 'cg/cg.dll': 'e084185a17fa1487ee5ffec5cff828d120278f03547838031e2d70bb2a8811fe', 'cg/game.py': '2558b876af5990b8648ecd8030456bf34bd0f8e4a7251e32a2c074f848a897da', 'cg/libcg-arm64.so': '95628f0bd20ac5e6f1e142c88cd52c9dc7361f9c371b5599f63b27c110892545', 'cg/libcg.dylib': 'da53cb331277cc8aa0a6181da28afeea28ae2f205372c234eca4a5c9cf3f4672', 'cg/libcg.so': 'c699020e086eede189c445c7abf9d891a60005ceec6eb91dccd5bbdcd3c0de22', 'cg/sim.py': 'e83ca04300990267cd04385fb98ded670b517dfceb481a2bfa63ff433e2946ff', 'cg/utils.py': '60f29665cee0a88525d6f0383bc45959a6262d16fe35ef380aece1e0ea13c49b'}
Checkpoint hashes {'A-J1': 'bde3146edfb059ba36c4b60a83d3bb6cb3

,policy,python_seed,reverse_player,status,result_code,steps,learned,fallback,terminal_returns,shaped_returns,invalid_responses,engine_rejections,error_message
0,A-J1,61000,False,passed,0,16,8,8,"[1, -1]","[1.0, -1.0]",0,0,
1,A-J1,61001,True,passed,1,50,28,22,"[-1, 1]","[-1.0, 1.0]",0,0,
2,A-J1,61002,False,passed,1,21,14,7,"[-1, 1]","[-1.0, 1.0]",0,0,
3,A-J1,61003,True,passed,1,30,19,11,"[-1, 1]","[-1.0, 1.0]",0,0,
4,A-J2,61000,False,passed,0,29,18,11,"[1, -1]","[1.0, -1.0]",0,0,
5,A-J2,61001,True,passed,0,68,40,28,"[1, -1]","[1.0, -1.0]",0,0,
6,A-J2,61002,False,passed,1,33,20,13,"[-1, 1]","[-1.0, 1.0]",0,0,
7,A-J2,61003,True,passed,1,45,27,18,"[-1, 1]","[-0.9999999999999999, 0.9999999999999999]",0,0,


In [4]:
if SDK_DIR:
    comparison = []
    for name, _, _ in specs:
        completed = [d for d in decisions if d["policy"] == name and "terminal_return" in d]
        if not completed:
            continue
        for scheme in ("terminal", "shaped"):
            comparison.append({"policy": name, "scheme": scheme, "accepted_decisions": len(completed),
                               "nonzero_rewards": sum(d[f"{scheme}_reward"] != 0 for d in completed),
                               "mean_absolute_decision_reward": mean(abs(d[f"{scheme}_reward"]) for d in completed),
                               "mean_absolute_return_shift": mean(abs(d[f"{scheme}_return"] - d["terminal_return"]) for d in completed)})
    if comparison:
        display(pd.DataFrame(comparison))
        examples = [next((d for d in decisions if d["source"] == source and "terminal_return" in d), None)
                    for source in ("learned", "heuristic")]
        display(pd.DataFrame([d for d in examples if d is not None]))
    else:
        print("No valid completed battles to compare.")

,policy,scheme,accepted_decisions,nonzero_rewards,mean_absolute_decision_reward,mean_absolute_return_shift
0,A-J1,terminal,117,4,0.034188,0.000000
1,A-J1,shaped,117,12,0.044872,0.022792
2,A-J2,terminal,175,4,0.022857,0.000000
3,A-J2,shaped,175,14,0.031905,0.025476


,source,probability,log_probability,learned_probability,step,seat,action,legal_option_count,prize_lead_seat0,terminal_reward,shaped_reward,terminal_return,shaped_return,policy,seed
0,learned,0.196667,-1.626245,0.2,3,0,[2],6,0.0,0,0.0,1,1.0,A-J1,61000
1,heuristic,NaN,NaN,NaN,0,0,[0],2,0.0,0,0.0,1,1.0,A-J1,61000
